In [1]:
import pandas as pd
import numpy as np
import requests

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================
# LOAD DATA
# =========================
BASE_URL = "https://api.openf1.org/v1"

def get_data(endpoint):
    return pd.DataFrame(requests.get(f"{BASE_URL}/{endpoint}").json())

weather = get_data("weather")
stints = get_data("stints")
pits = get_data("pit")
drivers = get_data("drivers")

weather_csv = pd.read_csv("weather.csv", parse_dates=["date"])

In [2]:
def add_team_name(df, drivers):
    driver_teams = (
        drivers[["session_key", "driver_number", "team_name"]]
        .dropna(subset=["team_name"])
        .drop_duplicates()
    )
    return df.merge(driver_teams, on=["session_key", "driver_number"], how="left")

In [3]:
def build_dataset(weather, stints, pits, drivers):

    data = []

    stints = add_team_name(stints, drivers)
    pits = add_team_name(pits, drivers)

    for sid in weather["session_key"].unique():

        stint_data = stints[stints["session_key"] == sid]
        pit_data = pits[pits["session_key"] == sid]

        for team in stint_data["team_name"].dropna().unique():

            team_stints = stint_data[stint_data["team_name"] == team]
            team_pits = pit_data[pit_data["team_name"] == team]

            first = team_stints["compound"].mode()
            target = team_stints["compound"].mode()

            data.append({
                "session_id": sid,
                "team": team,
                "pit_stops": len(team_pits),
                "target_compound": target.iloc[0] if not target.empty else "MEDIUM"
            })

    return pd.DataFrame(data)


df = build_dataset(weather, stints, pits, drivers)
df = df.dropna()

In [4]:
team_encoder = LabelEncoder()
compound_encoder = LabelEncoder()
stops_encoder = LabelEncoder()

team_encoder.fit(df["team"])
compound_encoder.fit(df["target_compound"])

df["team_enc"] = team_encoder.transform(df["team"])
df["target_compound_enc"] = compound_encoder.transform(df["target_compound"])
df["pit_stops_enc"] = stops_encoder.fit_transform(df["pit_stops"])

y_stops = df["pit_stops_enc"].values
y_tire = df["target_compound_enc"].values

num_stop_classes = len(stops_encoder.classes_)
num_tire_classes = len(compound_encoder.classes_)

In [5]:
def build_sequences(df, weather):

    X_seq = []

    for _, row in df.iterrows():

        sid = row["session_id"]

        w = weather[weather["session_key"] == sid]

        seq = w[
            ["air_temperature",
             "track_temperature",
             "humidity",
             "rainfall",
             "wind_speed"]
        ].values

        X_seq.append(seq)

    return X_seq


X_seq_raw = build_sequences(df, weather)

X_seq = pad_sequences(
    X_seq_raw,
    padding="post",
    dtype="float32"
)

In [6]:
X_train, X_test, y_stops_train, y_stops_test, y_tire_train, y_tire_test = train_test_split(
    X_seq,
    y_stops,
    y_tire,
    test_size=0.2,
    random_state=42
)

In [7]:
timesteps = X_seq.shape[1]
features = X_seq.shape[2]

# =====================
# STOPS MODEL
# =====================
inp = keras.Input(shape=(timesteps, features))
x = layers.GRU(64)(inp)
x = layers.Dense(64, activation="relu")(x)
out = layers.Dense(num_stop_classes, activation="softmax")(x)

stops_model = keras.Model(inp, out)

stops_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

stops_model.fit(
    X_train,
    y_stops_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

stops_model.save("stops_model.h5")

Epoch 1/30
77/77 [==============================] - 28s 71ms/step - loss: 3.0738 - accuracy: 0.1208 - val_loss: 2.7845 - val_accuracy: 0.0998
Epoch 2/30
77/77 [==============================] - 4s 58ms/step - loss: 2.7219 - accuracy: 0.1175 - val_loss: 2.7761 - val_accuracy: 0.0998
Epoch 3/30
77/77 [==============================] - 4s 58ms/step - loss: 2.7188 - accuracy: 0.1154 - val_loss: 2.7953 - val_accuracy: 0.0917
Epoch 4/30
77/77 [==============================] - 5s 67ms/step - loss: 2.7128 - accuracy: 0.1122 - val_loss: 2.7643 - val_accuracy: 0.0998
Epoch 5/30
77/77 [==============================] - 5s 60ms/step - loss: 2.6925 - accuracy: 0.1175 - val_loss: 2.7279 - val_accuracy: 0.0998
Epoch 6/30
77/77 [==============================] - 4s 57ms/step - loss: 2.6634 - accuracy: 0.1261 - val_loss: 2.6995 - val_accuracy: 0.0998
Epoch 7/30
77/77 [==============================] - 4s 57ms/step - loss: 2.6429 - accuracy: 0.1216 - val_loss: 2.6517 - val_accuracy: 0.1015
Epoch 8/30
7

In [8]:
inp2 = keras.Input(shape=(timesteps, features))

x2 = layers.GRU(64)(inp2)
x2 = layers.Dense(64, activation="relu")(x2)
out2 = layers.Dense(num_tire_classes, activation="softmax")(x2)

tire_model = keras.Model(inp2, out2)

tire_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

tire_model.fit(
    X_train,
    y_tire_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

tire_model.save("tire_model.h5")

Epoch 1/30
77/77 [==============================] - 8s 69ms/step - loss: 1.4694 - accuracy: 0.3590 - val_loss: 1.2808 - val_accuracy: 0.3781
Epoch 2/30
77/77 [==============================] - 4s 58ms/step - loss: 1.2925 - accuracy: 0.3856 - val_loss: 1.2824 - val_accuracy: 0.3781
Epoch 3/30
77/77 [==============================] - 5s 63ms/step - loss: 1.2905 - accuracy: 0.3811 - val_loss: 1.2863 - val_accuracy: 0.3781
Epoch 4/30
77/77 [==============================] - 5s 71ms/step - loss: 1.2948 - accuracy: 0.3684 - val_loss: 1.2833 - val_accuracy: 0.3781
Epoch 5/30
77/77 [==============================] - 5s 60ms/step - loss: 1.2846 - accuracy: 0.3880 - val_loss: 1.2680 - val_accuracy: 0.3535
Epoch 6/30
77/77 [==============================] - 5s 59ms/step - loss: 1.2767 - accuracy: 0.3885 - val_loss: 1.2585 - val_accuracy: 0.4124
Epoch 7/30
77/77 [==============================] - 5s 65ms/step - loss: 1.2729 - accuracy: 0.4077 - val_loss: 1.2637 - val_accuracy: 0.4124
Epoch 8/30
77

In [13]:
def get_user_input():
    team = input("Team: ")
    tire = input("Starting tire: ")
    quali = int(input("Qualifying position: "))
    return team, tire, quali

In [15]:
def build_inference_sequence(weather_csv):

    # you can optionally filter by full file or time window
    seq = weather_csv[
        ["air_temperature",
         "track_temperature",
         "humidity",
         "rainfall",
         "wind_speed"]
    ].values

    return pad_sequences(
        [seq],
        maxlen=X_seq.shape[1],
        padding="post",
        dtype="float32"
    )

In [16]:
team, tire, quali = get_user_input()

X_input = build_inference_sequence(weather_csv)

stops_class = np.argmax(stops_model.predict(X_input, verbose=0))
stops = stops_encoder.inverse_transform([stops_class])[0]

print("\n--- STRATEGY RESULT ---")
print("Recommended stops:", stops)

strategy = [tire]

for _ in range(stops_class):

    pred = tire_model.predict(X_input, verbose=0)
    tire_class = np.argmax(pred)

    strategy.append(
        compound_encoder.inverse_transform([tire_class])[0]
    )

for i, s in enumerate(strategy):
    print(f"Stint {i+1}: {s}")


--- STRATEGY RESULT ---
Recommended stops: 2
Stint 1: WET
Stint 2: SOFT
Stint 3: SOFT
